# Glüten — Gemma 4 E4B Marsh classifier (Unsloth QLoRA)

**Task:** Fine-tune `unsloth/gemma-4-E4B-it-unsloth-bnb-4bit` on IBDColEpi HE patches to classify a proxy Marsh grade.

**Honest framing — read this first.** IBDColEpi ships with pixel-level epithelium segmentation masks but **no Marsh / villous-atrophy / IEL annotations**. The labels trained below are weak-supervision proxies derived from epithelium mask coverage per patch (lower coverage ≈ more atrophy). They are methodologically principled but are **not pathologist-validated Marsh scores**. The hackathon writeup and the `/api/medgemma/marsh` endpoint surface this caveat explicitly.

**Target prize track:** Unsloth special technology.

**Runtime:** Kaggle free T4 (~30 min in fp16/bf16 mixed precision). Enable *Accelerator → GPU T4* and *Internet → On*.

**Attached data:** the private Kaggle dataset `gluten-ibdcolepi-sample` (uploaded from `data/structural/processed/patch-dataset-HE-sampled.zip` + `marsh_pseudo_labels-sampled.csv`).

**Setup quirks already accounted for in this notebook (don't undo them):**
- Pillow is left at Kaggle's pre-installed 12.2.0 — Unsloth installed with `--no-deps` so it can't downgrade Pillow and break the `_imaging` C extension.
- Vision tower is **frozen** (`finetune_vision_layers=False`). We only LoRA-tune the language head, which is enough to teach the model to map SigLIP visual features → the four Marsh labels.

In [1]:
%%capture
!pip install -q --no-deps "unsloth @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps "unsloth_zoo @ git+https://github.com/unslothai/unsloth-zoo.git"
!pip install -q --no-deps cut-cross-entropy
!pip install -q --upgrade transformers
!pip install -q bitsandbytes trl peft accelerate tifffile

import os
from kaggle_secrets import UserSecretsClient
os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
import transformers; print('Transformers:', transformers.__version__)
import PIL; print('Pillow:', PIL.__version__)

## 1 · Install Unsloth + deps

All `--no-deps` so we don't fight Kaggle's base Pillow / transformers / datasets versions. If this cell errors, do **Run → Factory reset** and rerun from cell 1.

In [2]:
%%capture
!pip install -q --no-deps "unsloth @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps "unsloth_zoo @ git+https://github.com/unslothai/unsloth-zoo.git"
!pip install -q --no-deps cut-cross-entropy
!pip install -q tifffile

import os
from kaggle_secrets import UserSecretsClient
os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
import PIL; print('Pillow:', PIL.__version__)  # should print 12.2.0

## 2 · Build pseudo-Marsh labels from epithelium masks

For each HE patch we compute `epi_frac = (mask > 0).mean()`, then quantile-bin into Marsh-0 / Marsh-1 / Marsh-3a / Marsh-3b. Marsh-2 is omitted (under-represented in literature; molecular layer dataset GSE164883 also skips it). ~30 s on Kaggle disk.

In [3]:
import pandas as pd, numpy as np
from PIL import Image
from pathlib import Path
from tqdm.auto import tqdm

DATA_ROOT = Path('/kaggle/input/datasets/faithogun/gluten-ibdcolepi-sample')
PATCHES = DATA_ROOT / 'patch-dataset-HE-sampled'
SEED_CSV = DATA_ROOT / 'marsh_pseudo_labels-sampled.csv'

seed = pd.read_csv(SEED_CSV)
print(seed.shape, seed['split'].value_counts())

def read_mask(label_path):
    return np.array(Image.open(PATCHES / label_path))
def read_image(image_path):
    return Image.open(PATCHES / image_path).convert('RGB')

seed['epi_frac'] = [float((read_mask(r['label_path']) > 0).mean())
                    for _, r in tqdm(seed.iterrows(), total=len(seed))]

q = seed['epi_frac'].quantile([0.25, 0.5, 0.75]).values
def bin_fn(f):
    if f >= q[2]: return 'Marsh-0'
    if f >= q[1]: return 'Marsh-1'
    if f >= q[0]: return 'Marsh-3a'
    return 'Marsh-3b'
seed['marsh_bin'] = seed['epi_frac'].apply(bin_fn)
print(seed['marsh_bin'].value_counts())
seed.to_csv('/kaggle/working/marsh_pseudo_labels_scored.csv', index=False)

(2551, 6) split
Trainset         2000
Testset           400
Validationset     151
Name: count, dtype: int64


  0%|          | 0/2551 [00:00<?, ?it/s]

marsh_bin
Marsh-3b    638
Marsh-0     638
Marsh-1     638
Marsh-3a    637
Name: count, dtype: int64


## 3 · Load Gemma 4 E4B in 4-bit, attach LoRA to the language head only

`finetune_vision_layers=False` is the load-bearing line. We freeze the SigLIP vision encoder and only LoRA-train the language part — that's enough to teach the model to output Marsh-0/1/3a/3b given the visual features SigLIP already extracts.

In [4]:
from unsloth import FastVisionModel
import torch

model, tokenizer = FastVisionModel.from_pretrained(
    'unsloth/gemma-4-E4B-it-unsloth-bnb-4bit',
    load_in_4bit=True,
    use_gradient_checkpointing='unsloth',
)
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers=False,
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    r=16, lora_alpha=32, lora_dropout=0.05,
    bias='none', random_state=42,
)
print("Loaded Gemma 4 E4B in 4-bit, vision tower frozen, LoRA on language head.")

/tmp/ipykernel_23/2543287403.py:1: UserWarning: WARNING: Unsloth should be imported before [transformers] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from unsloth import FastVisionModel


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.8: Fast Gemma4 patching. Transformers: 5.6.2.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

...

processor_config.json: 0.00B [00:00, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/32.2M [00:00<?, ?B/s]

Loaded Gemma 4 E4B in 4-bit, vision tower frozen, LoRA on language head.


## 4 · Build the image→label instruction dataset

In [5]:
from datasets import Dataset

INSTRUCTION = (
    'You are a histopathology assistant. Classify the Marsh grade of this HE-stained '
    'intestinal biopsy patch. Respond with exactly one of: Marsh-0, Marsh-1, Marsh-3a, Marsh-3b.'
)

def load_patch(image_path):
    return read_image(image_path).resize((224, 224))

def to_example(row):
    img = load_patch(row['image_path'])
    return {
        'messages': [
            {'role': 'user', 'content': [
                {'type': 'image', 'image': img},
                {'type': 'text', 'text': INSTRUCTION},
            ]},
            {'role': 'assistant', 'content': [{'type': 'text', 'text': row['marsh_bin']}]},
        ]
    }

train_df = seed[seed['split'] == 'Trainset'].sample(min(2000, (seed['split'] == 'Trainset').sum()), random_state=42)
val_df = seed[seed['split'] == 'Validationset']

train_ds = Dataset.from_list([to_example(r) for _, r in train_df.iterrows()])
val_ds = Dataset.from_list([to_example(r) for _, r in val_df.iterrows()])
print(train_ds, val_ds)

Dataset({
    features: ['messages'],
    num_rows: 2000
}) Dataset({
    features: ['messages'],
    num_rows: 151
})


## 5 · Train

Mixed precision (fp16 on T4). Expect ~30 min for 250 steps. If it OOMs, drop `per_device_train_batch_size` to 1 and bump `gradient_accumulation_steps` to 8.

**Don't sit and wait** — use *Save Version → Save & Run All (Commit)* to run this in the background once the notebook is correct.

In [6]:
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig
import os
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

FastVisionModel.for_training(model)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    data_collator=UnslothVisionDataCollator(model, tokenizer),
    train_dataset=train_ds,
    eval_dataset=val_ds,
    args=SFTConfig(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=8,
        num_train_epochs=1,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10,
        optim='adamw_8bit',
        weight_decay=0.01,
        lr_scheduler_type='cosine',
        seed=42,
        output_dir='/kaggle/working/gemma4-marsh-lora',
        report_to='none',
        remove_unused_columns=False,
        dataset_text_field='',
        dataset_kwargs={'skip_prepare_dataset': True},
        max_length=1024,
    ),
)
trainer.train()

Unsloth: Model does not have a default image size - using 512
Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
10,0.737948
20,0.057649
30,0.040310
40,0.032383
50,0.019301
60,0.009783
70,0.006944
80,0.005354
90,0.005328
100,0.003944


TrainOutput(global_step=250, training_loss=0.03859890955686569, metrics={'train_runtime': 1903.607, 'train_samples_per_second': 1.051, 'train_steps_per_second': 0.131, 'total_flos': 1.716001800042816e+16, 'train_loss': 0.03859890955686569})

## 6 · Evaluate on the held-out Testset split

In [7]:
FastVisionModel.for_inference(model)
from sklearn.metrics import classification_report, confusion_matrix
from transformers import AutoProcessor

processor = AutoProcessor.from_pretrained('unsloth/gemma-4-E4B-it-unsloth-bnb-4bit')

test_df = seed[seed['split'] == 'Testset'].sample(min(400, (seed['split'] == 'Testset').sum()), random_state=42)
preds, gold = [], []
for _, row in tqdm(test_df.iterrows(), total=len(test_df)):
    img = load_patch(row['image_path'])
    msgs = [{'role': 'user', 'content': [
        {'type': 'image', 'image': img},
        {'type': 'text', 'text': INSTRUCTION}]}]
    inputs = processor.apply_chat_template(
        msgs, add_generation_prompt=True, tokenize=True,
        return_dict=True, return_tensors='pt',
    ).to('cuda')
    out = model.generate(**inputs, max_new_tokens=8, temperature=0, do_sample=False)
    txt = processor.decode(out[0, inputs['input_ids'].shape[-1]:], skip_special_tokens=True).strip()
    preds.append(txt.split()[0] if txt else '')
    gold.append(row['marsh_bin'])

print(classification_report(gold, preds, labels=['Marsh-0','Marsh-1','Marsh-3a','Marsh-3b'], zero_division=0))
print(confusion_matrix(gold, preds, labels=['Marsh-0','Marsh-1','Marsh-3a','Marsh-3b']))

  0%|          | 0/400 [00:00<?, ?it/s]

              precision    recall  f1-score   support

     Marsh-0       0.76      0.77      0.76        81
     Marsh-1       0.57      0.68      0.62        96
    Marsh-3a       0.63      0.53      0.57        99
    Marsh-3b       0.85      0.84      0.85       124

    accuracy                           0.71       400
   macro avg       0.70      0.70      0.70       400
weighted avg       0.71      0.71      0.71       400

[[ 62  19   0   0]
 [ 20  65  11   0]
 [  0  29  52  18]
 [  0   1  19 104]]


## 7 · Save LoRA adapter

~134 MB. Download from Kaggle's file pane (right side, `/kaggle/working/gemma-marsh-lora/`). Then locally we wire it into the `/api/medgemma/marsh` route (route name kept for stable URL — it serves whichever Marsh classifier we have loaded).

In [ ]:
model.save_pretrained('/kaggle/working/gemma-marsh-lora')
tokenizer.save_pretrained('/kaggle/working/gemma-marsh-lora')
!ls -lh /kaggle/working/gemma-marsh-lora/